# Database Normalization Pipeline

Processes `unnormalized_orders.csv` through cleaning and three normal forms.

| Stage | Output |
|---|---|
| Clean | `output/cleaned_orders.xlsx` |
| 1NF   | `output/1nf.xlsx` |
| 2NF   | `output/2nf.xlsx` |
| 3NF   | `output/3nf.xlsx` |

In [1]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

def save(df_or_dict, filename):
    """Save a DataFrame or {sheet: DataFrame} dict to output/ as .xlsx."""
    path = OUT_DIR / filename
    sheets = df_or_dict if isinstance(df_or_dict, dict) else {"data": df_or_dict}
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        for name, df in sheets.items():
            df.to_excel(w, sheet_name=name, index=False)
    print(f"Saved: {path}  |  sheets: {list(sheets)}")

---
## Cleaning Data

- Rename columns to consistent `snake_case`
- Fix `date` format (`DD/MM/YYYY` → `YYYY-MM-DD`)
- Coerce numeric columns
- Drop duplicate rows
- EDA Check for corrupted text

In [2]:
df = pd.read_csv("data/unnormalized_orders.csv", dtype=str)
print(f"Raw: {df.shape}")

# ── Rename columns to snake_case ────────────────────────────────────────────
df.columns = ["order_id", "date", "customer_id", "customer_name",
              "product_id", "product_name", "price_rs", "quantity", "total_price_rs"]

# ── EDA: Deep Dive into Dirty Data ──────────────────────────────────────────
print("\n" + "="*45)
print("        EXPLORATORY DATA ANALYSIS")
print("="*45)

# 1. Show overall missing values
print("\n[1] Missing Values per Column:")
print(df.isna().sum())

# 2. Corrupted Product IDs (checking for decimals instead of whole numbers)
temp_prod_id = pd.to_numeric(df["product_id"], errors="coerce")
mask_corrupted_id = temp_prod_id.notna() & (temp_prod_id % 1 != 0)
print(f"\n[2] Corrupted Product IDs ({mask_corrupted_id.sum()} found):")
if mask_corrupted_id.any():
    print(df.loc[mask_corrupted_id, ["order_id", "product_id", "product_name"]].head())

# 3. Missing Product Data (Names or IDs missing entirely)
mask_missing_prod = df["product_name"].isna() | df["product_id"].isna()
print(f"\n[3] Missing Product Names/IDs ({mask_missing_prod.sum()} found):")
if mask_missing_prod.any():
    print(df.loc[mask_missing_prod, ["order_id", "product_id", "product_name"]].head())

# 4. Symbols on Aluminum Foil
print("\n[4] Weird Symbols Check ('Aluminum'):")
mask_alu = df["product_name"].str.contains("Aluminum", na=False)
if mask_alu.any():
    print(f"Raw string: {repr(df.loc[mask_alu, 'product_name'].values[0])}")
else:
    print("Aluminum foil entry not found.")

print("="*45 + "\n")

# ── Strip weird/non-ASCII symbols from text columns (e.g. ◆ in Aluminum) ────
for col in ["customer_name", "product_name"]:
    df[col] = df[col].str.replace(r"[^\x00-\x7F]+", " ", regex=True).str.strip()

if mask_alu.any():
    print(f"Cleaned 'Aluminum' entry: {df.loc[mask_alu, 'product_name'].values[0]}")
print("-----------------\n")

# ── Parse date ───────────────────────────────────────────────────────────────
df["date"] = pd.to_datetime(df["date"], dayfirst=True).dt.strftime("%Y-%m-%d")

# ── Coerce all numeric columns ───────────────────────────────────────────────
for col in ["order_id", "customer_id", "product_id", "price_rs", "quantity", "total_price_rs"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ── Drop rows where product_id is not a whole number ────────────────────────
# Corrupted rows appear as floats like 205.35, 205.36 ...
before = len(df)
df = df[df["product_id"].notna() & (df["product_id"] % 1 == 0)]
print(f"Dropped {before - len(df)} rows with non-integer product_id → {len(df)} rows")

# Convert id columns to integer now that floats are gone
for col in ["order_id", "customer_id", "product_id"]:
    df[col] = df[col].astype("Int64")

# ── Fill missing quantity with per-product median ────────────────────────────
qty_median = df.groupby("product_id")["quantity"].transform("median")
df["quantity"] = df["quantity"].fillna(qty_median.round())
print(f"quantity nulls remaining: {df['quantity'].isna().sum()}")

# ── Fill missing total_price_rs as price_rs x quantity ───────────────────────
mask = df["total_price_rs"].isna() & df["price_rs"].notna() & df["quantity"].notna()
df.loc[mask, "total_price_rs"] = df.loc[mask, "price_rs"] * df.loc[mask, "quantity"]
print(f"total_price_rs nulls remaining: {df['total_price_rs'].isna().sum()}")

# ── Drop exact duplicate rows ────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Dropped {before - len(df)} duplicate rows → final shape {df.shape}")

save(df, "cleaned_orders.xlsx")
df.head()

Raw: (310, 9)

        EXPLORATORY DATA ANALYSIS

[1] Missing Values per Column:
order_id           0
date               0
customer_id        0
customer_name      0
product_id         0
product_name      10
price_rs          10
quantity          15
total_price_rs    25
dtype: int64

[2] Corrupted Product IDs (10 found):
    order_id product_id product_name
300     1594     205.35          NaN
301     1157     205.36          NaN
302     1191     205.37          NaN
303     1128     205.38          NaN
304     1304     205.39          NaN

[3] Missing Product Names/IDs (10 found):
    order_id product_id product_name
300     1594     205.35          NaN
301     1157     205.36          NaN
302     1191     205.37          NaN
303     1128     205.38          NaN
304     1304     205.39          NaN

[4] Weird Symbols Check ('Aluminum'):
Raw string: '50.�Aluminum�foil'

Cleaned 'Aluminum' entry: 50. Aluminum foil
-----------------

Dropped 10 rows with non-integer product_id → 300 rows
q

,order_id,date,customer_id,customer_name,product_id,product_name,price_rs,quantity,total_price_rs
0,1236,2024-01-01,74,"Arjun, 40, Male",148,1. Apples,150.0,13.0,1950.0
1,1428,2024-01-01,54,"Priya, 33, Female",149,2. Bananas,40.0,1.0,40.0
2,1585,2024-01-01,71,"Liam, 42, Male",248,3. Oranges,80.0,3.0,240.0
3,1240,2024-01-02,56,"Emma, 53, Female",242,4. Grapes,200.0,15.0,3000.0
4,1311,2024-01-03,79,"Ayaan, 32, Male",299,5. Strawberries,300.0,18.0,5400.0


---
## First Normal Form (1NF)

**Violation:** `customer_name` stores `"name, age, gender"` in a single cell — not atomic.

**Fix:** Split into three atomic columns: `customer_name`, `customer_age`, `customer_gender`.

Also strip the numeric prefix from `product_name` (`"1. Apples"` → `"Apples"`).

In [3]:
df_1nf = df.copy()

# Split "name, age, gender" into atomic columns
split = df_1nf["customer_name"].str.split(",", n=2, expand=True)
df_1nf["customer_name"]   = split[0].str.strip()
df_1nf["customer_age"]    = pd.to_numeric(split[1].str.strip(), errors="coerce")
df_1nf["customer_gender"] = split[2].str.strip()

# Strip numeric prefix from product_name: "1. Apples" → "Apples"
df_1nf["product_name"] = df_1nf["product_name"].str.replace(r"^\d+\.\s*", "", regex=True)

# Reorder columns
df_1nf = df_1nf[["order_id", "date", "customer_id", "customer_name", "customer_age",
                  "customer_gender", "product_id", "product_name", "price_rs",
                  "quantity", "total_price_rs"]]

print("1NF shape:", df_1nf.shape)
save(df_1nf, "1nf.xlsx")
df_1nf.head()

1NF shape: (300, 11)
Saved: output\1nf.xlsx  |  sheets: ['data']


,order_id,date,customer_id,customer_name,customer_age,customer_gender,product_id,product_name,price_rs,quantity,total_price_rs
0,1236,2024-01-01,74,Arjun,40,Male,148,Apples,150.0,13.0,1950.0
1,1428,2024-01-01,54,Priya,33,Female,149,Bananas,40.0,1.0,40.0
2,1585,2024-01-01,71,Liam,42,Male,248,Oranges,80.0,3.0,240.0
3,1240,2024-01-02,56,Emma,53,Female,242,Grapes,200.0,15.0,3000.0
4,1311,2024-01-03,79,Ayaan,32,Male,299,Strawberries,300.0,18.0,5400.0


---
## Second Normal Form (2NF)

**Composite key:** `(order_id, product_id)`

**Partial dependencies** — attributes that depend on only *part* of the key:

| Attribute | Depends on |
|---|---|
| `date`, `customer_id`, `customer_name`, `customer_age`, `customer_gender` | `order_id` only |
| `product_name`, `price_rs` | `product_id` only |

**Fix:** Decompose into three tables (`orders`, `products`, `order_items`). 

*Note: Customer details strictly depend on the primary key `order_id` at this stage, so they remain in the `orders` table. We will resolve their transitive dependencies in 3NF.*

In [4]:
# depends on order_id only (resolving partial dependency)
orders_2nf = (df_1nf[["order_id", "date", "customer_id", "customer_name", "customer_age", "customer_gender"]]
          .drop_duplicates().reset_index(drop=True))

# depends on product_id only
products = (df_1nf[["product_id", "product_name", "price_rs"]]
            .drop_duplicates().reset_index(drop=True))

# depends on full composite key (order_id, product_id)
order_items = (df_1nf[["order_id", "product_id", "quantity", "total_price_rs"]]
               .drop_duplicates().reset_index(drop=True))

for name, tbl in [("orders_2nf", orders_2nf),
                  ("products", products), 
                  ("order_items", order_items)]:
    print(f"{name:12s}: {tbl.shape}  {list(tbl.columns)}")

save({"orders": orders_2nf, "products": products, "order_items": order_items}, "2nf.xlsx")

orders_2nf  : (297, 6)  ['order_id', 'date', 'customer_id', 'customer_name', 'customer_age', 'customer_gender']
products    : (50, 3)  ['product_id', 'product_name', 'price_rs']
order_items : (300, 4)  ['order_id', 'product_id', 'quantity', 'total_price_rs']
Saved: output\2nf.xlsx  |  sheets: ['orders', 'products', 'order_items']


---
## Third Normal Form (3NF)

**Transitive dependency** in `orders`:

`order_id → customer_id → customer_name, customer_age, customer_gender`

Customer attributes depend on `customer_id`, not directly on the primary key `order_id`.

**Fix:** Extract customer attributes into a separate `customers` table. The `orders` table retains only the foreign key `customer_id`.

In [5]:
# Extract transitively dependent columns into a customers table
customers = (orders_2nf[["customer_id", "customer_name", "customer_age", "customer_gender"]]
             .drop_duplicates().reset_index(drop=True))

# Remove the transitive dependencies from the orders table
orders_3nf = (orders_2nf[["order_id", "date", "customer_id"]]
              .drop_duplicates().reset_index(drop=True))

for name, tbl in [("orders", orders_3nf), ("customers", customers),
                  ("products", products), ("order_items", order_items)]:
    print(f"{name:12s}: {tbl.shape}  {list(tbl.columns)}")

save({"orders": orders_3nf, "customers": customers,
      "products": products, "order_items": order_items}, "3nf.xlsx")

print("\nAll done. Files in output/:")
for f in sorted(OUT_DIR.iterdir()):
    print(" ", f.name)

orders      : (297, 3)  ['order_id', 'date', 'customer_id']
customers   : (87, 4)  ['customer_id', 'customer_name', 'customer_age', 'customer_gender']
products    : (50, 3)  ['product_id', 'product_name', 'price_rs']
order_items : (300, 4)  ['order_id', 'product_id', 'quantity', 'total_price_rs']
Saved: output\3nf.xlsx  |  sheets: ['orders', 'customers', 'products', 'order_items']

All done. Files in output/:
  1nf.xlsx
  2nf.xlsx
  3nf.xlsx
  cleaned_orders.xlsx
